# Supply Chain Demand Forecasting — Exploratory Data Analysis

**Dataset:** DataCo Smart Supply Chain (180K+ order records)  
**Goal:** Understand demand patterns, seasonality, delivery risk, and prepare for feature engineering.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import load_raw, run_pipeline
from src.preprocessing import preprocess
from src.database import (
    total_demand_by_region, demand_by_category,
    monthly_demand_trend, top_products_by_demand,
    inventory_risk_query, late_delivery_summary
)

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

print('Libraries loaded.')

## 1. Load & Initial Inspection

In [ ]:
# Load raw and push to SQLite (idempotent)
raw = run_pipeline()
print(f'Raw shape: {raw.shape}')
raw.head(3)

In [ ]:
# Column-level null audit
null_pct = (raw.isna().sum() / len(raw) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
print('Columns with nulls (% missing):')
print(null_pct.to_string())

In [ ]:
raw.describe(include='all').T[['count','mean','std','min','max']].head(30)

## 2. Preprocessing

In [ ]:
df = preprocess(raw)
print(f'Clean shape: {df.shape}')
df.dtypes.tail(20)

## 3. SQL-layer Aggregations

In [ ]:
region_df = total_demand_by_region()
print('Demand by Region:')
display(region_df)

In [ ]:
monthly = monthly_demand_trend()
monthly['year_month'] = pd.to_datetime(monthly['year_month'])
display(monthly.tail(12))

## 4. Demand Trend Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Monthly quantity
axes[0].plot(monthly['year_month'], monthly['total_quantity'],
             color='#2196F3', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(monthly['year_month'], monthly['total_quantity'],
                     alpha=0.15, color='#2196F3')
axes[0].set_title('Monthly Total Order Quantity', fontweight='bold')
axes[0].set_ylabel('Units')

# Monthly revenue
axes[1].plot(monthly['year_month'], monthly['total_sales'],
             color='#FF5722', linewidth=2, marker='s', markersize=4)
axes[1].fill_between(monthly['year_month'], monthly['total_sales'],
                     alpha=0.15, color='#FF5722')
axes[1].set_title('Monthly Total Revenue ($)', fontweight='bold')
axes[1].set_ylabel('Revenue ($)')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(REPORTS / 'monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/monthly_trend.png')

In [ ]:
# Demand by region
fig, ax = plt.subplots(figsize=(10, 5))
region_sorted = region_df.sort_values('total_quantity', ascending=True)
bars = ax.barh(region_sorted['order_region'], region_sorted['total_quantity'],
               color=sns.color_palette('Blues_r', len(region_sorted)))
ax.set_xlabel('Total Units Ordered')
ax.set_title('Total Demand by Order Region', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'demand_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Category & Product Analysis

In [ ]:
cat_df = demand_by_category()
top_prod = top_products_by_demand(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category revenue
cat_sorted = cat_df.sort_values('total_sales', ascending=False).head(12)
axes[0].bar(cat_sorted['category_name'], cat_sorted['total_sales'],
            color=sns.color_palette('viridis', len(cat_sorted)))
axes[0].set_title('Revenue by Category (Top 12)', fontweight='bold')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)

# Top products
axes[1].barh(top_prod['product_name'][:15],
             top_prod['total_quantity'][:15],
             color='#FF5722', alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_xlabel('Total Units')
axes[1].set_title('Top 15 Products by Demand', fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS / 'category_product_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Delivery & Risk Analysis

In [ ]:
late_summ = late_delivery_summary()
inv_risk  = inventory_risk_query()

# Late delivery heatmap
pivot = late_summ.pivot_table(
    index='shipping_mode', columns='order_region',
    values='late_rate_pct', aggfunc='mean'
).fillna(0)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Late Rate %'})
ax.set_title('Late Delivery Rate (%) by Shipping Mode × Region', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'late_delivery_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shipping delay distribution
if 'actual_lead_time' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    sns.histplot(df['actual_lead_time'].dropna(), bins=30,
                 kde=True, ax=axes[0], color='#2196F3')
    axes[0].set_title('Actual Lead Time Distribution (days)', fontweight='bold')
    
    delay = df['days_shipping_real'] - df['days_shipping_scheduled']
    sns.histplot(delay.dropna(), bins=30, kde=True, ax=axes[1], color='#FF5722')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[1].set_title('Shipping Delay (Actual − Scheduled)', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(REPORTS / 'shipping_delay_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Seasonality Decomposition

In [ ]:
# Weekly demand aggregated across all products
df['week_start'] = df['order_date'].dt.to_period('W').dt.start_time
weekly_total = df.groupby('week_start')['order_quantity'].sum().reset_index()
weekly_total.columns = ['week_start', 'demand']
weekly_total = weekly_total.sort_values('week_start')

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Raw weekly
axes[0].plot(weekly_total['week_start'], weekly_total['demand'],
             color='#2196F3', linewidth=1.5, alpha=0.8)
axes[0].set_title('Weekly Demand — All Products', fontweight='bold')

# 4-week rolling mean
rolling_mean = weekly_total['demand'].rolling(4, center=True).mean()
axes[1].plot(weekly_total['week_start'], weekly_total['demand'],
             alpha=0.4, color='grey', linewidth=1)
axes[1].plot(weekly_total['week_start'], rolling_mean,
             color='#FF5722', linewidth=2, label='4-week MA')
axes[1].set_title('Trend — 4-Week Rolling Average', fontweight='bold')
axes[1].legend()

# Month-of-year box plot
monthly_box = df.groupby(['order_year', 'order_month'])['order_quantity'].sum().reset_index()
monthly_box.boxplot(column='order_quantity', by='order_month', ax=axes[2],
                    patch_artist=True)
axes[2].set_title('Demand Distribution by Month', fontweight='bold')
axes[2].set_xlabel('Month')
plt.suptitle('')  # suppress pandas auto-title

plt.tight_layout()
plt.savefig(REPORTS / 'seasonality_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/seasonality_decomposition.png')

## 8. Correlation Analysis

In [ ]:
num_cols = ['order_quantity', 'sales', 'order_profit', 'days_shipping_real',
            'days_shipping_scheduled', 'item_discount_rate', 'is_late',
            'item_profit_ratio', 'benefit_per_order']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key EDA Takeaways

| Observation | Implication |
|---|---|
| Demand peaks in Q4 (Oct–Dec) | Include `is_q4` and `is_holiday_season` as regressors |
| Western Europe + USCA are top demand regions | Regional encoding is necessary |
| Standard Class shipping has highest late-delivery rate | `shipping_delay` is a meaningful feature |
| Sporting Goods dominates category volume | Category-level forecasting is viable |
| Sales and order profit show moderate negative correlation with discount rate | Discount rate is a useful demand signal |
| Clear yearly seasonality, mild week-of-year patterns | Prophet's yearly seasonality + lag features will capture this |